In [1]:
import joblib
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
train = pd.read_csv("../db/letters_train_fixed.csv", header=None)
test = pd.read_csv("../db/letters_test_fixed.csv", header=None)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

train

Train shape: (88800, 785)
Test shape: (14800, 785)


,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
0,23,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,16,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,15,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,23,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88795,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
88796,21,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
88797,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
88798,23,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
import string

# A-Z
letters = list(string.ascii_uppercase)

# если классы 1-26
label_map = {i + 1: letters[i] for i in range(26)}
label_map

{1: 'A',
 2: 'B',
 3: 'C',
 4: 'D',
 5: 'E',
 6: 'F',
 7: 'G',
 8: 'H',
 9: 'I',
 10: 'J',
 11: 'K',
 12: 'L',
 13: 'M',
 14: 'N',
 15: 'O',
 16: 'P',
 17: 'Q',
 18: 'R',
 19: 'S',
 20: 'T',
 21: 'U',
 22: 'V',
 23: 'W',
 24: 'X',
 25: 'Y',
 26: 'Z'}

In [4]:
train

,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
0,23,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,16,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,15,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,23,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88795,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
88796,21,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
88797,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
88798,23,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
y_train = train[0]
print("Number of classes:", y_train.nunique())
print(y_train.value_counts().sort_index())

Number of classes: 26
0
1     3396
2     3396
3     3419
4     3398
5     3437
6     3394
7     3385
8     3424
9     3428
10    3402
11    3438
12    3415
13    3402
14    3365
15    3408
16    3430
17    3435
18    3419
19    3392
20    3436
21    3419
22    3422
23    3423
24    3437
25    3453
26    3427
Name: count, dtype: int64


In [6]:
print("Missing values:", train.isnull().sum().sum())

Missing values: 0


In [7]:
y_train = train[0]
y_letters = y_train.map(label_map)
X_train = train.drop(0, axis=1)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (88800, 784)
y_train shape: (88800,)


In [8]:
# CV
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=150)),
    ("model", LogisticRegression(
        C=20,
        max_iter=3000,
        solver="lbfgs"
    ))
])


# Получаем предсказания через CV
y_pred = cross_val_predict(
    pipe,
    X_train,
    y_train,
    cv=skf,
    n_jobs=-1
)

# Метрики
print("Accuracy:", accuracy_score(y_train, y_pred))
print(classification_report(y_train, y_pred))

Accuracy: 0.7145157657657658
              precision    recall  f1-score   support

           1       0.60      0.57      0.58      3396
           2       0.73      0.72      0.73      3396
           3       0.80      0.83      0.82      3419
           4       0.66      0.65      0.66      3398
           5       0.79      0.75      0.77      3437
           6       0.73      0.75      0.74      3394
           7       0.59      0.49      0.54      3385
           8       0.67      0.69      0.68      3424
           9       0.57      0.60      0.59      3428
          10       0.73      0.78      0.75      3402
          11       0.69      0.68      0.69      3438
          12       0.56      0.59      0.58      3415
          13       0.85      0.87      0.86      3402
          14       0.66      0.66      0.66      3365
          15       0.81      0.87      0.84      3408
          16       0.80      0.84      0.82      3430
          17       0.62      0.61      0.62      343

In [9]:
X_test = test.drop(0, axis=1)
y_test = test[0]

In [10]:
pipe.fit(X_train, y_train)
y_pred_test = pipe.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test, target_names=letters, zero_division=0))

Test accuracy: 0.7089864864864864
              precision    recall  f1-score   support

           A       0.66      0.62      0.64       800
           B       0.80      0.78      0.79       800
           C       0.80      0.81      0.80       800
           D       0.74      0.64      0.69       800
           E       0.83      0.76      0.79       800
           F       0.82      0.74      0.78       800
           G       0.64      0.47      0.54       800
           H       0.73      0.68      0.70       800
           I       0.67      0.63      0.65       800
           J       0.81      0.75      0.78       800
           K       0.74      0.68      0.71       800
           L       0.62      0.60      0.61       800
           M       0.89      0.89      0.89       800
           N       0.73      0.64      0.68       800
           O       0.85      0.89      0.87       800
           P       0.83      0.86      0.84       800
           Q       0.64      0.62      0.63    